In [ ]:
import pandas as pd
from wsi_stats import WSIStatsCache

# Path to WSIs (root dir)
ROOT_DIR = "//regsj.intern/appl/Deep_Visual_Proteomics"

# Path to cache file
CACHE_FILE = "wsi_cache_christine.pkl"

# Load/reload WSI stats cache
wsi_cache = WSIStatsCache(ROOT_DIR, CACHE_FILE)
df_wsi = wsi_cache.main(reload=True)

In [ ]:
# Path to pathology metadata
df_path = "D:\DATA\initial_cleaning.csv"
df_pathology = pd.read_csv(df_path)

In [ ]:
# Check for WSI with no associated data
missing_data = df_wsi[
    (df_wsi["file_size"].isna() | (df_wsi["file_size"] == 0)) |
    (df_wsi["data_folder_size"].isna() | (df_wsi["data_folder_size"] == 0))
]
print("WSI with missing data: ", missing_data)

In [ ]:
# TODO Look into missing data - file error, name mismatch, other

In [ ]:
# Drop rows with no associated data
rows_to_drop = missing_data[missing_data.any(axis=1)].index
df_no_missing = df_wsi.drop(index=rows_to_drop)

print(f"Original rows: {len(df_wsi)} \nRows with missing data: {len(rows_to_drop)} \nAfter dropping: {len(df_no_missing)}")

In [ ]:
print(df_pathology["rekvnr"].head())
print(df_no_missing["rekvnr"].head())

In [ ]:
# Convert rekvnr to int
df_pathology["rekvnr"] = pd.to_numeric(df_pathology["rekvnr"], errors="coerce").astype("Int64")
df_no_missing["rekvnr"] = pd.to_numeric(df_no_missing["rekvnr"], errors="coerce").astype("Int64")


In [ ]:
# Set of all rekvnr in pathology df and wsi df 
rekvnr_pathology = set(df_pathology["rekvnr"])
rekvnr_wsi = set(df_no_missing["rekvnr"])

# Calculate overlap of rekvnr
overlap = rekvnr_pathology & rekvnr_wsi
only_in_pathology = rekvnr_pathology - rekvnr_wsi
only_in_wsi = rekvnr_wsi - rekvnr_pathology

print(f"rekvnr in both: {len(overlap)}")
print(f"rekvnr only in pathology: {len(only_in_pathology)}")
print(f"rekvnr only in WSI: {len(only_in_wsi)}")

In [ ]:
count_in_wsi = df_no_missing["rekvnr"].isin(overlap).sum()
print(f"Number of rows in WSI with overlapping rekvnr: {count_in_wsi}")

counts_per_rekvnr = df_no_missing[df_no_missing["rekvnr"].isin(overlap)]["rekvnr"].value_counts()
print(counts_per_rekvnr)

In [ ]:
from ocr_labels import SlideLabelOCR

# Filter df_no_missing to keep only slides whose rekvnr is in "only_in_wsi"
df_only_in_wsi = df_no_missing[df_no_missing["rekvnr"].isin(only_in_wsi)].copy()

# Extract rekvnr from label 
ocr_rekvnr = []
ocr_conf = []

for _, row in df_only_in_wsi.iterrows():
    slide_path = row["filename"]
    ocr = SlideLabelOCR(slide_path)
    _, text, conf = ocr.results

    ocr_rekvnr.append(text)
    ocr_conf.append(conf)

df_only_in_wsi["ocr_rekvnr"] = ocr_rekvnr
df_only_in_wsi["ocr_conf"] = ocr_conf

In [ ]:
# Filter by confidence > 90
df_high_conf = df_only_in_wsi[df_only_in_wsi["ocr_conf"] > 90]

# Convert OCR rekvnr to a set
ocr_rekvnr_set = set(df_high_conf["ocr_rekvnr"])

# Find overlap with pathology rekvnr
overlap_ocr = ocr_rekvnr_set & rekvnr_pathology

print(f"rekvnr only in WSI: {len(only_in_wsi)}")
print(f"Total high-confidence OCR: {len(ocr_rekvnr_set)}")
print(f"Overlap with pathology rekvnr: {len(overlap_ocr)}")

In [ ]:
# Union of both overlap sets
combined_overlap = overlap | overlap_ocr

# Count slides with overlapping rekvnr
count_wsi = df_no_missing["rekvnr"].isin(combined_overlap).sum()
print(f"Number of slides with overlapping rekvnr: {count_wsi}")

# Subset pathology DataFrame to all overlapping rekvnr
df_overlap = df_pathology[df_pathology["rekvnr"].isin(combined_overlap)].copy()

# Count WSI files per rekvnr
wsi_counts = df_no_missing["rekvnr"].value_counts()
df_overlap["wsi_count"] = df_overlap["rekvnr"].map(wsi_counts).fillna(0).astype(int)

print(df_overlap.head())

In [ ]:
# Create a mapping from rekvnr to list of filenames
filenames_per_rekvnr = df_no_missing.groupby("rekvnr").apply(
    lambda g: list(g.index),
    include_groups=False
)

# Add the list of filenames as a new column in df_overlap
df_overlap["wsi filenames"] = df_overlap["rekvnr"].map(filenames_per_rekvnr)

print(df_overlap[["rekvnr", "wsi count", "wsi filenames"]].head())

In [ ]:
# Save to csv
output_file = "D:\DATA\overlapping_rekvnr.csv"
df_overlap.to_csv(output_file, index=False)

print(f"Saved DataFrame to {output_file}")